# Run whitebox QA and collect taxonomy reports

Runs Testable whitebox for selected branches, exports **HTML only**, and saves under:

`taxonomy_reports/<L2 group>/<BRANCH>/<BRANCH>_<timestamp>/taxonomy-gate.html`

## Parameters

In [ ]:
import json
import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "lib" / "sa_qa.py").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from lib import metrics
from lib.registry import load_registry, iter_branches
from lib.sa_qa import run_taxonomy_batch, verify_taxonomy_file, load_env
from lib.qa_reports import git_commit_report

ENV_FILE = str(ROOT / ".env.local")
VERSION = "2.6"
REFRESH_BRANCHES = True
SKIP_VERIFY = False
COMMIT_EACH_REPORT = True

# --- Scope: single metric × 4 branch types (override for partial runs) ---
BRANCHES = [
    "SA_DOV_Bug_2.6",
    "SA_DOV_BugFX_2.6",
    "SA_DOV_TCC_2.6",
    "SA_DOV_CC_2.6",
]

# Set USE_MANIFEST=True to run all branches from build_manifest.json instead
USE_MANIFEST = False
MANIFEST = ROOT / "build_manifest.json"

if USE_MANIFEST and MANIFEST.is_file():
    branches = json.loads(MANIFEST.read_text(encoding="utf-8")).get("branches", [])
    print(f"Loaded {len(branches)} branches from build_manifest.json")
else:
    branches = list(BRANCHES)
    print(f"Planned {len(branches)} branches: {', '.join(branches)}")
branches_csv = ",".join(branches)

## Run QA batch

In [ ]:
rc, batch_dir = run_taxonomy_batch(
    env_file=ENV_FILE,
    branches_csv=branches_csv,
    refresh_branches=REFRESH_BRANCHES,
    export_html=True,
    html_only=True,
)
if rc != 0:
    raise RuntimeError(f"Batch failed rc={rc}")
print("Reports root:", batch_dir)

## Save reports on main + verify

In [ ]:
import os
from lib.metrics import infer_from_branch_name
from lib.sa_qa import iter_taxonomy_html_paths

summary = []
for html_path in iter_taxonomy_html_paths(batch_dir):
    run_dir = os.path.dirname(html_path)
    branch = os.path.basename(os.path.dirname(run_dir))
    tech, metric, btype, _ = infer_from_branch_name(branch)
    rel = os.path.relpath(html_path, ROOT).replace("\\", "/")
    commit_sha = ""
    if COMMIT_EACH_REPORT:
        commit_sha = git_commit_report(str(ROOT), rel, branch)
    ok, detail = (True, "skipped") if SKIP_VERIFY else verify_taxonomy_file(html_path, metric, btype)
    summary.append({"branch": branch, "path": rel, "verify": ok, "commit": commit_sha})
    print(branch, "OK" if ok else "FAIL", rel)
    if not ok and not SKIP_VERIFY:
        print(detail)

print(f"\nProcessed {len(summary)} HTML report(s)")

## Summary table

In [ ]:
for row in summary:
    print(row)